# Vault RAG — Ulepszona wersja

## Streszczenie

W tej lekcji poprawiliśmy wyniki Vault RAG z **3/9 → 4/9** trafnych odpowiedzi przez trzy niezależne usprawnienia architektury oraz poprawki danych w vaulcie.

### Ulepszenia architektury pipeline'u

**1. Wyszukiwanie przez iloczyn (Step 2)**

Lesson 9 używała sumy zbiorów (OR) — każdy produkt pojawiający się w DOWOLNEJ stronie powierzchni lub kategorii trafiał do kandydatów. Powodowało to fałszywe trafienia (np. mydło do rąk wchodziło przez kategorię, mimo że nie pasuje do zapytania o łazienkę).

Rozwiązanie: wymagamy, by kandydat pojawiał się zarówno na stronie powierzchni **i** na stronie kategorii z intencji:
```
Stare: OR(powierzchnie) ∪ OR(kategorie)
Nowe:  OR(powierzchnie) ∩ OR(kategorie)   [z fallback do sumy gdy wynik < 3]
```

**2. Filtr bezpieczeństwa LLM (nowy Step 2.5)**

Dedykowane, tanie wywołanie LLM, które czyta strony kandydatów i zwraca `excluded_ids` — produkty, których sekcja `Odradzane powierzchnie` lub ostrzeżenia BHP wykluczają je z opisanego kontekstu.

Dlaczego LLM zamiast regex Python?
- Nazwy w `Odradzane` nie zawsze odpowiadają dokładnie nazwom powierzchni z intencji (np. `kamień naturalny` vs `marmur`)
- LLM wychwytuje niezgodności chemiczne sugerowane przez opis (kwas na marmur, nawet bez jawnego linku)

```
filter_unsafe(pytanie, kandydaci) → (bezpieczni_kandydaci, excluded_ids, usage)
```

**3. Ustrukturyzowany CoT w prompcie rekomendacji (RECOMMEND_SYSTEM_3)**

Stary prompt mieszał zasady bezpieczeństwa z oceną formy i kontekstu. Nowy prompt rozdziela odpowiedzialności:
- **Ochrona przed halucynacjami**: "Opieraj się W 100% na faktach — nie wymyślaj przeznaczenia produktu"
- **Oś A — FORMA I TYP**: dopasowanie formy (żel/piana/spray), ręczne vs maszynowe, "np. X lub Y = PRZYKŁADY, uwzględnij wszystkie"
- **Oś B — KONTEKST I SKALA**: produkty profesjonalne są OK jeśli pasują; wyklucz tylko czysto przemysłowe
- **Permisywny fallback**: "Jeśli produkt ogólnie pasuje i nie ma przeciwwskazań — uwzględnij go"
- Bezpieczeństwo NIE jest powtarzane — zostało już obsłużone przez `filter_unsafe`

---

### Poprawki danych w vaulcie

Analiza nieudanych zapytań ujawniła błędy w strukturze vaultu — produkty były pomijane przez pipeline, bo nie były połączone z właściwymi stronami powierzchni lub kategorii:

| Produkt | Problem | Poprawka |
|---------|---------|---------|
| Zurada Czysta Łazienka (11) | Nie było na stronie `marmur.md`; opis nie wspominał o bezpieczeństwie dla kamienia | Dodano do `marmur.md`; opis zaktualizowany: "bezpieczny dla marmuru i blatów kamiennych" |
| Zurada Sanibłysk (10), Zurada Sanit Żel (12) | Odradzane nie zawierało `marmur` — filtr bezpieczeństwa ich nie usuwał | Dodano `marmur` i `kamień naturalny` do sekcji Odradzane |
| Zurada Szklany Blask (2), Zurada Szkło Błysk (39) | Brak `dla-domu` w kategoriach; brak na stronach `szyby.md` i `lustra.md` | Dodano `dla-domu`, `mycie-szyb-i-luster`; dodano do szyby i lustra |
| Zurada Kryształ (41) | Brak na stronach `szyby.md` i `lustra.md` | Dodano do obu stron powierzchni |
| Zurada Owocowa Mgła (217) | Opis tylko "owocowy zapach" — LLM nie łączył z wariantem winogronowym/grejpfrutowym | Dodano konkretne owoce (jabłko, gruszka, brzoskwinia) i link do serii zapachowej |

---

## Diagram pipeline'u

```
pytanie klienta
  → [LLM] extract_intent        → powierzchnie[], kategorie[], metody[]
  → [Python] lookup_candidates   → iloczyn(powierzchnie ∩ kategorie), fallback do sumy gdy < 3
  → [LLM] filter_unsafe          → bezpieczni_kandydaci (excluded_ids usunięte)
  → [LLM] recommend              → product_ids[] z chain_of_thought
```

3 wywołania LLM na pytanie (intencja + bezpieczeństwo + rekomendacja).

In [59]:
%pip install instructor

Note: you may need to restart the kernel to use updated packages.


In [60]:
import os
import re
import time
import json
from pathlib import Path
from datetime import datetime
from typing import List

from openai import OpenAI
from pydantic import BaseModel, Field
import instructor

OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY", "Brak klucza")
#MODEL    = "openai/gpt-5.4-mini"
MODEL = "google/gemini-3.1-flash-lite"
BASE_URL = "https://openrouter.ai/api/v1"

BASE_DIR       = Path(".")
VAULT_DIR      = BASE_DIR / "zurada_vault"
QUESTIONS_FILE = BASE_DIR / "../5_navie_rag_extended_data/extended_golden_questions.json"

_raw_client = OpenAI(api_key=OPENROUTER_API_KEY, base_url=BASE_URL)
client = instructor.from_openai(_raw_client, mode=instructor.Mode.JSON)

with open(QUESTIONS_FILE, encoding="utf-8") as f:
    questions = json.load(f)

print(f"Model: {MODEL}")
print(f"Vault: {VAULT_DIR.resolve()}")
print(f"Questions: {len(questions)}")
print(f"API key: {'OK' if OPENROUTER_API_KEY != 'Brak klucza' else 'BRAK'}")

Model: google/gemini-3.1-flash-lite
Vault: /Users/adamzurada/projects/ai-knowledge/rag-product-recommend/lessons/11_vault_rag_improvements/zurada_vault
Questions: 9
API key: OK


---
## MAP_KNOWLEDGE + Pydantic models

**Course pattern (lesson 3.4):** the `Field(description=...)` IS the prompt for each field.
Instead of injecting 300+ names as a flat list into the prompt body, we put **semantic descriptions with examples** directly in the field definition. Instructor passes these to the model as part of the JSON schema.

Three techniques applied:
- `MethodEnum` — closed set of 22 valid method names; pydantic rejects anything outside it and instructor retries
- Rich `Field(description=...)` — top-12 surfaces/categories with semantic hints, so "toaleta" maps to "toalety"
- Explicit defaults — `"Domyślnie zwróć pustą listę"` prevents the model from guessing when uncertain

In [61]:
MAP_KNOWLEDGE = """
Vault Zurada to baza wiedzy produktów czyszczących, zorganizowana jako mapa połączonych stron.

Struktura vaultu:
- Powierzchnie/  — strony dla każdej powierzchni (np. toalety, podłogi, szkło)
- Kategorie/     — strony dla każdej kategorii (np. wc, zele-do-wc, odkamienianie)
- Metody/        — metody mycia (np. ręczne, myjnia, pianowanie)
- Produkty/      — pełne opisy produktów (frontmatter: product_id, ph, metody, kategorie)

Zasady odpowiedzi:
- Odpowiadaj WYŁĄCZNIE na podstawie dostarczonych stron produktów
- Nie wymyślaj cech produktów spoza dostarczonych danych
- Odróżniaj formy (żel vs piana), przeznaczenie (domowe vs przemysłowe), metodę (ręczne vs maszynowe)
"""


class ProductCandidate(BaseModel):
    product_id: int
    reason: str = Field(description="Jedno zdanie uzasadnienia")


class RecommendModel(BaseModel):
    chain_of_thought: List[str]         = Field(default_factory=list)
    product_ids: List[ProductCandidate] = Field(default_factory=list)
    excluded_product_ids: List[ProductCandidate] = Field(default_factory=list)


print("MAP_KNOWLEDGE and RecommendModel defined")
print("IntentModel will be defined after vault index is loaded")

MAP_KNOWLEDGE and RecommendModel defined
IntentModel will be defined after vault index is loaded


---
## Step 0 — Build vault index

Parse the vault once at startup: build maps of `surface/category → product names`.
No LLM needed here — pure file parsing.

In [62]:
def parse_product_links(md_text: str) -> list[str]:
    return re.findall(r'\[\[Produkty/([^|\]]+)[|\]]', md_text)


def parse_frontmatter_field(md_text: str, field: str):
    m = re.search(rf'^{field}:\s*(.+)$', md_text, re.MULTILINE)
    return m.group(1).strip() if m else None


name_to_id: dict[str, int] = {}
for f in (VAULT_DIR / "Produkty").glob("*.md"):
    text = f.read_text(encoding="utf-8")
    pid  = parse_frontmatter_field(text, "product_id")
    if pid:
        name_to_id[f.stem] = int(pid)

surfaces_index: dict[str, list[str]] = {}
for f in (VAULT_DIR / "Powierzchnie").glob("*.md"):
    surfaces_index[f.stem] = parse_product_links(f.read_text(encoding="utf-8"))

categories_index: dict[str, list[str]] = {}
for f in (VAULT_DIR / "Kategorie").glob("*.md"):
    categories_index[f.stem] = parse_product_links(f.read_text(encoding="utf-8"))

methods_index: dict[str, list[str]] = {}
for f in (VAULT_DIR / "Metody").glob("*.md"):
    methods_index[f.stem] = parse_product_links(f.read_text(encoding="utf-8"))

print(f"Products: {len(name_to_id)}  Surfaces: {len(surfaces_index)}  "
      f"Categories: {len(categories_index)}  Methods: {len(methods_index)}")

Products: 262  Surfaces: 307  Categories: 285  Methods: 22


In [63]:
import enum as _enum

# --- helpers ---

def _page_description(path: Path) -> str:
    text  = path.read_text(encoding="utf-8")
    parts = text.split("---", 2)
    body  = parts[2] if len(parts) >= 3 else text
    for line in body.strip().split("\n"):
        line = line.strip()
        if line and not line.startswith("#") and not line.startswith("-"):
            return line[:100]
    return ""


def _make_enum(name: str, keys: list[str]) -> type:
    """Build a str-enum from vault page names.
    Pydantic validates returned values against the enum;
    instructor retries automatically when the model returns an invalid name.
    """
    return _enum.Enum(
        name,
        {k.upper().replace(" ", "_").replace("-", "_").replace("(", "").replace(")", ""): k
         for k in sorted(keys)},
        type=str,
    )


# --- Three closed-set enums (lesson 3.4 pattern) ---
# Pro: exact-match validation + automatic retry on invalid values
# Cost: instructor embeds the full enum schema in the prompt
#   MethodEnum   22 values  →  negligible token cost
#   SurfaceEnum  307 values →  ~1,500 extra tokens per call
#   CategoryEnum 285 values →  ~1,400 extra tokens per call

MethodEnum   = _make_enum("MethodEnum",   list(methods_index.keys()))
SurfaceEnum  = _make_enum("SurfaceEnum",  list(surfaces_index.keys()))
CategoryEnum = _make_enum("CategoryEnum", list(categories_index.keys()))

_method_desc = "; ".join(
    f"{m.value}: {_page_description(VAULT_DIR / 'Metody' / f'{m.value}.md')}"
    for m in MethodEnum
)


# --- IntentModel with all three enums ---
class IntentModel(BaseModel):
    surfaces: List[SurfaceEnum] = Field(
        default_factory=list,
        description=(
            "Powierzchnie fizyczne wymienione lub sugerowane w pytaniu. "
            "Domyślnie zwróć pustą listę jeśli brak jednoznacznego dopasowania."
        ),
    )
    categories: List[CategoryEnum] = Field(
        default_factory=list,
        description=(
            "Kategorie produktów pasujące do pytania. "
            "Domyślnie zwróć pustą listę jeśli brak jednoznacznego dopasowania."
        ),
    )
    methods: List[MethodEnum] = Field(
        default_factory=list,
        description=(
            "Metody aplikacji wymagane w pytaniu. "
            "Wybierz tylko jeśli pytanie jawnie wskazuje metodę. "
            "Domyślnie zwróć pustą listę. "
            f"Dostępne: {_method_desc}"
        ),
    )
    reasoning_steps: List[str] = Field(default_factory=list)


print(f"SurfaceEnum:  {len(list(SurfaceEnum))} values")
print(f"CategoryEnum: {len(list(CategoryEnum))} values")
print(f"MethodEnum:   {len(list(MethodEnum))} values")
print("All three use closed-set validation → instructor retries on invalid names")

SurfaceEnum:  307 values
CategoryEnum: 285 values
MethodEnum:   22 values
All three use closed-set validation → instructor retries on invalid names


---
## Step 1 — Extract intent from the question

A small, cheap LLM call that asks: *what surfaces, categories and methods does this question relate to?*  
The LLM returns keys that map directly to vault page names.

In [64]:
INTENT_SYSTEM_3 = """Jesteś analitykiem zapytań dla bazy wiedzy produktów czyszczących Zurada.

{MAP_KNOWLEDGE}

Na podstawie pytania klienta zidentyfikuj, które strony vaultu są potrzebne.
ZASADY KRYTYCZNE:
1. UŻYWAJ WYŁĄCZNIE WARTOŚCI Z LISTY. Nie wymyślaj własnych słów (np. nie używaj słowa "muszla", jeśli nie ma go w dostępnym Enumie).
2. ZACHOWAJ ORYGINALNĄ PISOWNIĘ Z ENUMÓW. Jeśli kategoria w schemacie nie ma polskich znaków (np. "reczne-mycie-naczyn"), zwróć ją DOKŁADNIE w takiej formie, bez "ę", "ń" itp.
3. KATEGORYZUJ POPRAWNIE: 
   - 'surfaces' to fizyczne obiekty (np. 'toalety', 'blaty'). Nigdy nie wkładaj tu procesów ani typów chemii.
   - 'categories' to grupy chemii domowej/warsztatowej.
Wybierz od 1 do maksymalnie 6 najbardziej pasujących pozycji per pole."""

def extract_intent(question: str) -> tuple[IntentModel, object]:
    intent, completion = client.chat.completions.create_with_completion(
        model=MODEL,
        messages=[
            {"role": "system", "content": INTENT_SYSTEM_3.format(MAP_KNOWLEDGE=MAP_KNOWLEDGE)},
            {"role": "user",   "content": question},
        ],
        response_model=IntentModel,
        max_tokens=4000,
        temperature=0,
    )
    return intent, completion.usage


demo_q = questions[0]
intent, usage_demo = extract_intent(demo_q["question"])
print(f"Question:   {demo_q['question']}")
print(f"Surfaces:   {intent.surfaces}")
print(f"Categories: {intent.categories}")
print(f"Methods:    {[m.value for m in intent.methods]}")
print(f"Reasoning:  {intent.reasoning_steps}")
print(f"Usage:      prompt={usage_demo.prompt_tokens}  completion={usage_demo.completion_tokens}  total={usage_demo.total_tokens}")

Question:   Potrzebuję zwykłego, gęstego żelu do codziennego mycia toalety w domu, żeby usunąć trochę kamienia i odświeżyć muszlę.
Surfaces:   [<SurfaceEnum.TOALETY: 'toalety'>]
Categories: [<CategoryEnum.ZELE_DO_WC: 'zele-do-wc'>, <CategoryEnum.CHEMIA_DOMOWA: 'chemia-domowa'>, <CategoryEnum.HIGIENA_TOALET: 'higiena-toalet'>]
Methods:    ['ręczne']
Reasoning:  ['Klient szuka produktu do codziennego mycia toalety w domu.', "Wymagana forma to 'żel', co wskazuje na kategorię 'zele-do-wc'.", "Zastosowanie domowe sugeruje kategorię 'chemia-domowa' oraz 'higiena-toalet'.", 'Usuwanie kamienia sugeruje potrzebę produktu o właściwościach odkamieniających, typowych dla żeli do WC.', 'Metoda aplikacji w domu jest ręczna.']
Usage:      prompt=7865  completion=147  total=8012


---
## Step 2 — Look up vault pages

Use the intent to find candidate products.  
No LLM — pure index lookup using the vault's linked structure.

In [65]:
def lookup_candidates(intent: IntentModel) -> set[str]:
    """Step 2: vault lookup with intersection logic.

    OR-union (lesson 9) admits false positives: a product that only appears in one
    category page but has nothing to do with the user's surfaces is still included.

    Improvement: when the intent contains both surfaces AND categories, require
    candidates to appear in at least one surface page AND at least one category page
    (intersection). Fall back to union only when the intersection would be empty.
    """
    from_surfaces: set[str] = set()
    for surface in intent.surfaces:
        from_surfaces.update(surfaces_index.get(surface.value, []))

    from_categories: set[str] = set()
    for cat in intent.categories:
        from_categories.update(categories_index.get(cat.value, []))

    from_methods: set[str] = set()
    for method in intent.methods:
        from_methods.update(methods_index.get(method.value, []))

    if from_surfaces and from_categories:
        intersection = from_surfaces & from_categories
        # Fallback to union if intersection is too small (< 3 products) to avoid killing recall
        candidates = intersection if len(intersection) >= 3 else (from_surfaces | from_categories)
    elif from_surfaces:
        candidates = from_surfaces
    elif from_categories:
        candidates = from_categories
    else:
        candidates = from_methods

    return candidates


candidates = lookup_candidates(intent)
print(f"Candidate products ({len(candidates)}):")
candidates_id = sorted(name_to_id[c] for c in candidates if c in name_to_id)
print(candidates_id)

Candidate products (7):
[10, 12, 29, 79, 80, 82, 139]


---
## Step 2.5 — LLM safety pre-filter

Before the full recommend call, a small focused LLM call reads the candidate pages and
identifies products whose `Odradzane powierzchnie` or BHP warnings make them unsafe for
the user's context.

**Why LLM rather than Python regex?**
- Odradzane page names don't always exactly match intent surface names (e.g. `kamień naturalny` vs `marmur`)  
- Can catch chemical incompatibilities implied by the description even if not listed in odradzane
- Same vault data, focused auditor role → cheap call, high-confidence exclusions

In [66]:
def load_product_pages(product_names: set[str]) -> str:
    """Read vault pages for the given product names and return them as one text block."""
    pages = []
    for name in sorted(product_names):
        path = VAULT_DIR / "Produkty" / f"{name}.md"
        if path.exists():
            pages.append(path.read_text(encoding="utf-8"))
    return "\n\n---\n\n".join(pages)


SAFETY_SYSTEM = """Jesteś ekspertem chemicznym oceniającym bezpieczeństwo środków czyszczących.

Otrzymasz strony produktów z bazy Zurada oraz pytanie klienta.

Twoje jedyne zadanie: zidentyfikuj produkty, których NIE NALEŻY stosować w opisanym kontekście.

Wyklucz produkt (zwróć jego product_id) TYLKO jeśli zachodzi jeden z poniższych warunków:
1. Jego sekcja "Odradzane powierzchnie" zawiera powierzchnię wymienioną lub sugerowaną w pytaniu.
2. Skład/opis wskazuje na KONKRETNĄ chemiczną niezgodność z kontekstem (np. silny kwas na marmur, chlor/podchloryn na aluminium).
3. Ostrzeżenia BHP wprost wykluczają opisany sposób użycia.

ZASADY "NIGDY NIE WYKLUCZAJ":
- Nigdy nie wykluczaj produktu tylko dlatego, że jest silny, profesjonalny lub przeznaczony do intensywnego czyszczenia.
- Nigdy nie wykluczaj produktu "na wszelki wypadek" — tylko wyraźna przesłanka z pkt 1–3 uzasadnia wykluczenie.
- Nigdy nie dopowiadaj własnych kryteriów np. "niebezpieczny dla dzieci" lub "niebezpieczny dla zwierząt", jeśli nie ma tego wprost w danych produktu.

Jeśli żaden produkt nie narusza kryteriów 1–3 — zwróć pustą listę excluded_ids."""


SAFETY_SYSTEM_2 = """Jesteś ekspertem chemicznym oceniającym bezpieczeństwo środków czyszczących.

Otrzymasz strony produktów z bazy Zurada oraz pytanie klienta.

Twoje jedyne zadanie: zidentyfikuj produkty NIEBEZPIECZNE dla opisanego kontekstu — nie filtruj
pod kątem trafności czy dopasowania (to robi osobny krok).

Wyklucz produkt (zwróć jego product_id) TYLKO jeśli zachodzi jeden z poniższych warunków:
1. Jego sekcja "Odradzane powierzchnie" zawiera powierzchnię WPROST wymienioną w pytaniu klienta.
2. Skład/opis wskazuje na KONKRETNĄ, BEZPOŚREDNIĄ niezgodność chemiczną z kontekstem
   (przykłady: silny kwas na marmur lub kamień naturalny, podchloryn na aluminium lub tkaniny,
   silnie żrący zasadowy środek na lakierowane powierzchnie).
3. Ostrzeżenia BHP tego produktu wprost wykluczają opisany sposób użycia.

WAŻNE — produkty higieny osobistej i mydła:
Mydła w płynie, żele do mycia rąk, szampony i środki do higieny osobistej (kategorie vault:
'mydla-w-plynie', 'higiena-rak', 'higiena-osobista', 'mycie-rak') są z definicji bezpieczne
dla skóry i typowych powierzchni łazienkowych. Wyklucz je wyłącznie gdy kryterium 1 jest
spełnione — tzn. gdy ich "Odradzane powierzchnie" zawiera konkretną powierzchnię z pytania.
Nie wykluczaj ich na podstawie kryterium 2 ani 3.

ZASADY OGÓLNE:
- Ten filtr ocenia wyłącznie BEZPIECZEŃSTWO, nie trafność produktu.
- Nie wykluczaj produktu tylko dlatego, że jest silny, profesjonalny lub nie najlepiej pasuje.
- Nie dodawaj własnych kryteriów (np. "za drogi", "dla profesjonalistów", "nieodpowiedni zapach").
- Gdy wątpliwość — zachowaj produkt (nie wykluczaj)."""


class SafetyFilterModel(BaseModel):
    excluded_ids: List[int] = Field(
        default_factory=list,
        description="product_id wartości produktów niebezpiecznych dla opisanego kontekstu"
    )
    reasoning: List[str] = Field(default_factory=list)


def filter_unsafe(question: str, candidates: set[str]) -> tuple[set[str], SafetyFilterModel, object]:
    """Step 2.5: LLM safety audit — returns (safe_candidates, safety_result, usage)."""
    product_context = load_product_pages(candidates)
    user_msg = f"Strony produktów:\n\n{product_context}\n\n---\n\nPytanie klienta: {question}"

    safety, completion = client.chat.completions.create_with_completion(
        model=MODEL,
        messages=[
            {"role": "system", "content": SAFETY_SYSTEM_2},
            {"role": "user",   "content": user_msg},
        ],
        response_model=SafetyFilterModel,
        max_tokens=1000,
        temperature=0,
    )

    id_to_name = {v: k for k, v in name_to_id.items()}
    excluded_names = {id_to_name[pid] for pid in safety.excluded_ids if pid in id_to_name}
    safe_candidates = candidates - excluded_names

    return safe_candidates, safety, completion.usage


# Demo
demo_q = questions[0]
intent, _ = extract_intent(demo_q["question"])
candidates = lookup_candidates(intent)
safe_candidates, safety_result, usage_safety = filter_unsafe(demo_q["question"], candidates)

print(f"Before filter: {len(candidates)} candidates")
print(f"After filter:  {len(safe_candidates)} candidates")
print(f"Excluded IDs:  {safety_result.excluded_ids}")
print(f"Usage:         prompt={usage_safety.prompt_tokens}  total={usage_safety.total_tokens}")
for r in safety_result.reasoning:
    print(f"  {r}")

Before filter: 7 candidates
After filter:  7 candidates
Excluded IDs:  []
Usage:         prompt=6677  total=6746
  Żaden z produktów nie jest niebezpieczny w kontekście codziennego mycia toalety w domu, ponieważ pytanie nie wskazuje na użycie na powierzchniach wrażliwych (takich jak marmur, kamień naturalny czy aluminium), które są wymienione w sekcjach 'Odradzane powierzchnie' niektórych produktów.


In [67]:
RECOMMEND_SYSTEM_3 = """Jesteś ekspertem ds. środków czystości firmy Zurada.

{MAP_KNOWLEDGE}

Poniżej otrzymasz strony produktów, które przeszły już filtr bezpieczeństwa — możesz zakładać,
że żaden z nich nie jest chemicznie niebezpieczny dla opisanego kontekstu.

Twoim zadaniem jest wybranie produktów, które najlepiej odpowiadają na pytanie klienta.
Opieraj się W 100% na faktach z dostarczonych stron — nie wymyślaj przeznaczenia produktu.

Oceń każdy produkt według 2 osi, wpisując rozumowanie do chain_of_thought PRZED podaniem wyników.

A) FORMA I TYP
   Czy forma produktu (żel / piana / spray / proszek / koncentrat) i jego główne przeznaczenie
   odpowiadają dokładnie temu, czego szuka klient? Zwróć uwagę na:
   - formę fizyczną (żel vs piana vs spray)
   - przeznaczenie (mycie ręczne vs maszynowe/zmywarka, czyszczenie vs dezynfekcja)
   - gdy klient pyta "np. winogrona lub grejpfrut" — to tylko PRZYKŁADY; uwzględnij WSZYSTKIE
     produkty tego samego typu (inne owoce też pasują)

B) KONTEKST I SKALA
   Produkty profesjonalne (HoReCa, firmy sprzątające) rekomenduj jeśli pasują do opisanego
   problemu — nie wykluczaj ich tylko dlatego, że są "profesjonalne" lub "agresywne" lub "silne".
   Wyklucz jedynie środki dedykowane wyłącznie skalą przemysłową (np. beczki 200L, linie produkcyjne).

Jeśli produkt ogólnie pasuje do opisanego zastosowania i nie ma wyraźnych przeciwwskazań — uwzględnij go.
Jeśli żaden nie pasuje — zwróć pustą listę product_ids."""


def recommend(question: str, product_context: str) -> tuple[RecommendModel, object]:
    """Step 4: structured CoT recommend — safety already handled upstream."""
    system = RECOMMEND_SYSTEM_3.format(MAP_KNOWLEDGE=MAP_KNOWLEDGE)
    user_msg = f"Strony produktów (po filtrze bezpieczeństwa):\n\n{product_context}\n\n---\n\nPytanie klienta: {question}"

    result, completion = client.chat.completions.create_with_completion(
        model=MODEL,
        messages=[
            {"role": "system", "content": system},
            {"role": "user",   "content": user_msg},
        ],
        response_model=RecommendModel,
        max_tokens=4000,
        temperature=0,
    )
    return result, completion.usage


result, usage_rec = recommend(demo_q["question"], load_product_pages(safe_candidates))

returned_ids = [c.product_id for c in result.product_ids]
print(f"Question:  {demo_q['question']}")
print(f"Expected:  {demo_q['expectedProductIds']}")
print(f"Returned:  {returned_ids}")
print(f"Usage:     prompt={usage_rec.prompt_tokens}  total={usage_rec.total_tokens}")
for c in result.product_ids:
    print(f"  [{c.product_id}] {c.reason}")
print("Products excluded for safety reasons:")    
for c in result.excluded_product_ids:
    print(f"  [{c.product_id}] {c.reason}")

Question:  Potrzebuję zwykłego, gęstego żelu do codziennego mycia toalety w domu, żeby usunąć trochę kamienia i odświeżyć muszlę.
Expected:  [12, 82, 139]
Returned:  [139, 82, 12]
Usage:     prompt=6996  total=7616
  [139] Jest to dedykowany żel do użytku domowego, idealny do codziennego czyszczenia toalet i usuwania kamienia.
  [82] Ekologiczny, skoncentrowany żel przeznaczony do codziennego utrzymania czystości w sanitariatach.
  [12] Zagęszczony żel do codziennego czyszczenia i odkamieniania, skuteczny w usuwaniu osadów.
Products excluded for safety reasons:
  [79] Produkt zawiera chlor i działa wybielająco, co wykracza poza potrzebę 'zwykłego' mycia.
  [80] Produkt jest przeznaczony do usuwania bardzo silnego, wieloletniego kamienia i rdzy, co jest zbyt agresywne do codziennego mycia.
  [10] Jest to pianka, a klient wyraźnie poszukiwał żelu.
  [29] Jest to pianka, a klient wyraźnie poszukiwał żelu.


In [68]:
def vault_rag_pipeline(question: str) -> tuple[list[int], dict]:
    """Improved pipeline: intent → intersection lookup → LLM safety filter → recommend."""
    t0 = time.perf_counter()

    intent, usage_intent = extract_intent(question)

    # Step 2: intersection-based lookup
    candidates = lookup_candidates(intent)

    # Step 2.5: LLM safety pre-filter
    safe_candidates, safety_result, usage_safety = filter_unsafe(question, candidates)

    product_context = load_product_pages(safe_candidates)

    # Step 4: recommend from the safe candidate set
    rec, usage_rec = recommend(question, product_context)

    elapsed = time.perf_counter() - t0

    returned_ids = [c.product_id for c in rec.product_ids]
    stats = {
        "intent":             {"surfaces": [s.value for s in intent.surfaces],
                               "categories": [c.value for c in intent.categories],
                               "methods": [m.value for m in intent.methods]},
        "n_candidates":       len(candidates),
        "n_safe_candidates":  len(safe_candidates),
        "excluded_ids":       safety_result.excluded_ids,
        "n_tokens_est":       len(product_context) // 4,
        "prompt_tokens":      usage_intent.prompt_tokens  + usage_safety.prompt_tokens  + usage_rec.prompt_tokens,
        "completion_tokens":  usage_intent.completion_tokens + usage_safety.completion_tokens + usage_rec.completion_tokens,
        "total_tokens":       usage_intent.total_tokens   + usage_safety.total_tokens   + usage_rec.total_tokens,
        "time_seconds":       round(elapsed, 2),
    }
    return returned_ids, stats


results = []

for q in questions:
    qid      = q["id"]
    expected = set(q["expectedProductIds"])

    try:
        returned_ids, stats = vault_rag_pipeline(q["question"])
        returned = set(returned_ids)
        error = None
    except Exception as e:
        returned, stats, error = set(), {
            "n_candidates": 0, "n_safe_candidates": 0, "excluded_ids": [],
            "intent": {}, "time_seconds": 0,
            "prompt_tokens": 0, "completion_tokens": 0, "total_tokens": 0,
        }, str(e)

    hit     = expected & returned
    missed  = expected - returned
    extra   = returned - expected
    correct = returned == expected

    results.append({
        "id":                qid,
        "difficulty":        q.get("difficulty", ""),
        "expected":          sorted(expected),
        "returned":          sorted(returned),
        "hit":               sorted(hit),
        "missed":            sorted(missed),
        "extra":             sorted(extra),
        "correct":           correct,
        "n_candidates":      stats.get("n_candidates", 0),
        "n_safe_candidates": stats.get("n_safe_candidates", 0),
        "excluded_ids":      stats.get("excluded_ids", []),
        "n_tokens_est":      stats.get("n_tokens_est", 0),
        "prompt_tokens":     stats.get("prompt_tokens", 0),
        "completion_tokens": stats.get("completion_tokens", 0),
        "total_tokens":      stats.get("total_tokens", 0),
        "time_seconds":      stats.get("time_seconds", 0),
        "intent":            stats.get("intent", {}),
        "error":             error,
    })

    status = "✓" if correct else "✗"
    print(f"[{status}] Q{qid:02d} [{q['difficulty']:4s}] "
          f"candidates={stats.get('n_candidates',0):3d}→{stats.get('n_safe_candidates',0):3d}  "
          f"excluded={stats.get('excluded_ids',[])}  "
          f"tokens={stats.get('total_tokens',0):6,}  "
          f"time={stats.get('time_seconds',0):.1f}s  "
          f"expected={sorted(expected)}  returned={sorted(returned)}")

correct_total = sum(r["correct"] for r in results)
print(f"\n{correct_total}/{len(results)} exact matches  "
      f"avg time={sum(r['time_seconds'] for r in results)/len(results):.1f}s  "
      f"avg tokens={sum(r['total_tokens'] for r in results)/len(results):,.0f}")

[✗] Q01 [easy] candidates=  7→  4  excluded=[79, 10, 29]  tokens=19,324  time=5.3s  expected=[12, 82, 139]  returned=[82, 139]
[✗] Q08 [easy] candidates= 10→  9  excluded=[137]  tokens=20,419  time=5.2s  expected=[136, 137, 189, 190]  returned=[136, 189, 190]
[✓] Q09 [easy] candidates=  4→  2  excluded=[197, 244]  tokens=14,103  time=4.3s  expected=[233, 257]  returned=[233, 257]
[✓] Q10 [easy] candidates= 18→ 17  excluded=[215]  tokens=30,439  time=4.4s  expected=[213, 217, 218]  returned=[213, 217, 218]


KeyboardInterrupt: 

In [69]:
from concurrent.futures import ThreadPoolExecutor, as_completed

N_RUNS      = 8
MAX_WORKERS = 9          # one concurrent thread per question
OUTPUT_FILE = BASE_DIR / "improved_vault_rag_runs.json"


def _run_once(q: dict, run_idx: int) -> tuple[int, int, dict]:
    """Execute one pipeline run; returns (question_id, run_idx, result_dict)."""
    try:
        returned_ids, stats = vault_rag_pipeline(q["question"])
        result = {
            "run":               run_idx,
            "returned_ids":      returned_ids,
            "model":             MODEL,
            "n_candidates":      stats["n_candidates"],
            "n_safe_candidates": stats["n_safe_candidates"],
            "excluded_ids":      stats["excluded_ids"],
            "prompt_tokens":     stats["prompt_tokens"],
            "completion_tokens": stats["completion_tokens"],
            "total_tokens":      stats["total_tokens"],
            "time_seconds":      stats["time_seconds"],
            "intent":            stats["intent"],
        }
    except Exception as e:
        result = {"run": run_idx, "error": str(e)}
    return q["id"], run_idx, result


# Build all (question, run) tasks upfront
tasks = [(q, run_idx) for q in questions for run_idx in range(1, N_RUNS + 1)]

# Collect raw results keyed by question id
from collections import defaultdict
raw: dict[int, list] = defaultdict(list)

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
    futures = {pool.submit(_run_once, q, run_idx): (q["id"], run_idx)
               for q, run_idx in tasks}

    for future in as_completed(futures):
        qid, run_idx, result = future.result()
        raw[qid].append(result)
        status = "✓" if not result.get("error") else "✗"
        print(f"  {status} Q{qid:02d} run {run_idx}")


# Aggregate per question
repeated = []

for q in questions:
    qid  = q["id"]
    runs = sorted(raw[qid], key=lambda r: r["run"])

    times  = [r["time_seconds"] for r in runs if "time_seconds" in r]
    tokens = [r["total_tokens"]  for r in runs if "total_tokens"  in r]

    avg_time   = round(sum(times)  / len(times),  2) if times  else None
    avg_tokens = round(sum(tokens) / len(tokens), 0) if tokens else None

    repeated.append({
        "id":                   qid,
        "question":             q["question"],
        "expectedProductIds":   q["expectedProductIds"],
        "expectedNoProductIds": q["expectedNoProductIds"],
        "difficulty":           q.get("difficulty", ""),
        "avg_time_seconds":     avg_time,
        "avg_total_tokens":     avg_tokens,
        "runs":                 runs,
    })

    expected_set = set(q["expectedProductIds"])
    n_correct    = sum(1 for r in runs if set(r.get("returned_ids", [])) == expected_set)
    print(f"Q{qid:02d} [{q['difficulty']:4s}] {n_correct}/{N_RUNS} correct  "
          f"avg={avg_time:.1f}s  avg_tokens={avg_tokens:,.0f}")


with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump({
        "timestamp": datetime.now().isoformat(),
        "model":     MODEL,
        "approach":  "improved_vault_rag",
        "n_runs":    N_RUNS,
        "questions": repeated,
    }, f, ensure_ascii=False, indent=2)

print(f"\nSaved: {OUTPUT_FILE}")

  ✓ Q08 run 1
  ✓ Q01 run 5
  ✓ Q01 run 4
  ✓ Q01 run 1
  ✓ Q01 run 8
  ✓ Q01 run 7
  ✓ Q01 run 3
  ✓ Q01 run 2
  ✓ Q08 run 2
  ✓ Q08 run 4
  ✓ Q09 run 1
  ✓ Q08 run 6
  ✓ Q08 run 7
  ✓ Q08 run 3
  ✓ Q08 run 8
  ✓ Q01 run 6
  ✓ Q09 run 2
  ✓ Q09 run 4
  ✓ Q09 run 6
  ✓ Q09 run 5
  ✓ Q09 run 3
  ✓ Q09 run 7
  ✓ Q09 run 8
  ✓ Q08 run 5
  ✓ Q10 run 1
  ✓ Q10 run 3
  ✓ Q10 run 4
  ✓ Q10 run 5
  ✓ Q10 run 8
  ✓ Q10 run 7
  ✓ Q02 run 1
  ✓ Q02 run 2
  ✓ Q02 run 4
  ✓ Q02 run 3
  ✓ Q10 run 2
  ✓ Q10 run 6
  ✓ Q02 run 6
  ✓ Q02 run 5
  ✓ Q03 run 1
  ✓ Q03 run 4
  ✓ Q03 run 2
  ✓ Q03 run 3
  ✓ Q03 run 5
  ✓ Q02 run 7
  ✓ Q03 run 7
  ✓ Q03 run 6
  ✓ Q02 run 8
  ✓ Q04 run 1
  ✓ Q04 run 2
  ✓ Q03 run 8
  ✓ Q04 run 4
  ✓ Q04 run 3
  ✓ Q04 run 5
  ✓ Q04 run 7
  ✓ Q04 run 6
  ✓ Q04 run 8
  ✓ Q05 run 1
  ✓ Q05 run 5
  ✓ Q05 run 2
  ✓ Q05 run 4
  ✓ Q05 run 3
  ✓ Q05 run 8
  ✓ Q05 run 6
  ✓ Q05 run 7
  ✓ Q06 run 1
  ✓ Q06 run 2
  ✓ Q06 run 3
  ✓ Q06 run 4
  ✓ Q06 run 6
  ✓ Q06 run 5
  ✓ Q06 run 8
  ✓ Q0